# Intermediate: Advanced Data Processing

Master advanced data handling techniques for better model performance!

Topics covered:
- Custom data augmentation
- Handling imbalanced datasets  
- Mixed precision data loading
- Efficient data pipelines
- On-the-fly preprocessing
- Caching and prefetching
- Multi-modal data handling

## Why Advanced Data Processing?

```
Poor Data Pipeline:          Optimized Pipeline:
GPU: ████░░░░░░░░░░░░        GPU: ████████████████
CPU: ░░░░████░░░░░░░░        CPU: ████████████████
(GPU waiting for data!)      (GPU always busy!)

Result: 50% GPU utilization  Result: 95% GPU utilization
```

**Good data processing = Faster training + Better models**


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


## 1. Custom Data Augmentation

Beyond basic transforms - advanced augmentation techniques.


In [ ]:
# Custom augmentation function
class CustomAugmentation:
    def __init__(self, prob=0.5):
        self.prob = prob
    
    def __call__(self, img):
        if torch.rand(1) < self.prob:
            # Custom augmentation: Add random noise
            noise = torch.randn_like(img) * 0.1
            img = img + noise
            img = torch.clamp(img, 0, 1)
        return img

# Advanced augmentation pipeline
advanced_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    transforms.ToTensor(),
    CustomAugmentation(prob=0.5),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

print("Advanced augmentation pipeline created!")
print(advanced_transform)


## 2. Mixup and CutMix

Advanced augmentation techniques that mix images.


In [ ]:
# Mixup: Mix two images and labels
def mixup_data(x, y, alpha=1.0):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

# CutMix: Cut and paste patches between images
def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    # Get random box
    W = x.size(2)
    H = x.size(3)
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    
    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)
    
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]
    
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (W * H))
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

print("Mixup and CutMix functions defined!")
print()
print("Usage in training loop:")
print("  inputs, targets_a, targets_b, lam = mixup_data(inputs, targets)")
print("  loss = lam * criterion(outputs, targets_a) + (1-lam) * criterion(outputs, targets_b)")


## 3. Handling Imbalanced Datasets

Techniques for dealing with class imbalance.


In [ ]:
# Simulate imbalanced dataset
class_counts = torch.tensor([5000, 500, 50])  # Highly imbalanced
print("Class distribution:")
for i, count in enumerate(class_counts):
    print(f"  Class {i}: {count} samples ({100*count/class_counts.sum():.1f}%)")
print()

# Solution 1: Weighted Random Sampler
class_weights = 1. / class_counts
sample_weights = class_weights[torch.tensor([0]*5000 + [1]*500 + [2]*50)]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("Weighted Random Sampler created!")
print("  Each class will be sampled proportionally to inverse frequency")
print()

# Solution 2: Weighted Loss
def get_class_weights(class_counts):
    total = class_counts.sum()
    weights = total / (len(class_counts) * class_counts)
    return weights

weights = get_class_weights(class_counts)
print("Class weights for loss function:")
for i, w in enumerate(weights):
    print(f"  Class {i}: {w:.3f}")

criterion = nn.CrossEntropyLoss(weight=weights.to(device))
print("\nWeighted CrossEntropyLoss created!")
print()

# Solution 3: Oversampling minority classes
from torch.utils.data import ConcatDataset, Subset

def oversample_minority_classes(dataset, target_samples_per_class):
    # Get class indices
    targets = [dataset[i][1] for i in range(len(dataset))]
    class_indices = {}
    for idx, target in enumerate(targets):
        if target not in class_indices:
            class_indices[target] = []
        class_indices[target].append(idx)
    
    # Oversample
    balanced_indices = []
    for class_id, indices in class_indices.items():
        # Repeat indices to reach target count
        repeats = target_samples_per_class // len(indices) + 1
        balanced_indices.extend(indices * repeats)
        balanced_indices = balanced_indices[:target_samples_per_class]
    
    return Subset(dataset, balanced_indices)

print("Oversampling function defined!")


## 4. Efficient Data Loading with Prefetching


In [ ]:
# Prefetch data to GPU
class DataPrefetcher:
    def __init__(self, loader, device):
        self.loader = iter(loader)
        self.device = device
        self.stream = torch.cuda.Stream() if device.type == 'cuda' else None
        self.preload()
    
    def preload(self):
        try:
            self.next_input, self.next_target = next(self.loader)
        except StopIteration:
            self.next_input = None
            self.next_target = None
            return
        
        if self.stream is not None:
            with torch.cuda.stream(self.stream):
                self.next_input = self.next_input.to(self.device, non_blocking=True)
                self.next_target = self.next_target.to(self.device, non_blocking=True)
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.next_input is None:
            raise StopIteration
        
        if self.stream is not None:
            torch.cuda.current_stream().wait_stream(self.stream)
        
        input = self.next_input
        target = self.next_target
        self.preload()
        return input, target

print("Data prefetcher created!")
print("\nUsage:")
print("  prefetcher = DataPrefetcher(train_loader, device)")
print("  for input, target in prefetcher:")
print("      # Training loop")
print()
print("Benefits:")
print("  ✓ Overlaps data transfer with computation")
print("  ✓ Reduces GPU idle time")
print("  ✓ Can improve training speed by 10-30%")


## 5. Caching for Expensive Preprocessing


In [ ]:
# Cache preprocessed data
class CachedDataset(Dataset):
    def __init__(self, base_dataset, transform=None):
        self.base_dataset = base_dataset
        self.transform = transform
        self.cache = {}
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        if idx in self.cache:
            img, label = self.cache[idx]
        else:
            img, label = self.base_dataset[idx]
            # Cache the base data
            self.cache[idx] = (img.copy() if hasattr(img, 'copy') else img, label)
        
        # Apply transforms (not cached, for randomness)
        if self.transform:
            img = self.transform(img)
        
        return img, label

print("Cached dataset class created!")
print()
print("When to cache:")
print("  ✓ Loading from disk is slow (e.g., large images)")
print("  ✓ Enough RAM available")
print("  ✗ Dataset is too large for memory")
print("  ✗ Using extensive random augmentation")


## 6. Multi-Modal Data Handling


In [ ]:
# Example: Image + Text dataset
class MultiModalDataset(Dataset):
    def __init__(self, image_paths, captions, image_transform=None):
        self.image_paths = image_paths
        self.captions = captions
        self.image_transform = image_transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load image
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.image_transform:
            image = self.image_transform(image)
        
        # Process text (simplified)
        caption = self.captions[idx]
        # In practice, you'd tokenize and encode the text
        
        return {'image': image, 'caption': caption}

# Custom collate function for variable-length data
def multi_modal_collate_fn(batch):
    images = torch.stack([item['image'] for item in batch])
    captions = [item['caption'] for item in batch]
    return {'images': images, 'captions': captions}

print("Multi-modal dataset class created!")
print()
print("Usage:")
print("  dataset = MultiModalDataset(image_paths, captions, transform)")
print("  loader = DataLoader(dataset, batch_size=32, collate_fn=multi_modal_collate_fn)")


## 7. Online Hard Example Mining (OHEM)


In [ ]:
# Focus on hard examples during training
class OHEMDataLoader:
    def __init__(self, dataset, model, batch_size, keep_ratio=0.7):
        self.dataset = dataset
        self.model = model
        self.batch_size = batch_size
        self.keep_ratio = keep_ratio
        self.criterion = nn.CrossEntropyLoss(reduction='none')
    
    def get_hard_examples(self):
        self.model.eval()
        all_losses = []
        
        with torch.no_grad():
            for idx in range(len(self.dataset)):
                img, label = self.dataset[idx]
                img = img.unsqueeze(0).to(device)
                label = torch.tensor([label]).to(device)
                
                output = self.model(img)
                loss = self.criterion(output, label)
                all_losses.append((idx, loss.item()))
        
        # Sort by loss (descending)
        all_losses.sort(key=lambda x: x[1], reverse=True)
        
        # Keep hardest examples
        num_keep = int(len(all_losses) * self.keep_ratio)
        hard_indices = [idx for idx, _ in all_losses[:num_keep]]
        
        return hard_indices

print("OHEM DataLoader class created!")
print()
print("How it works:")
print("  1. Evaluate model on all samples")
print("  2. Compute loss for each sample")
print("  3. Keep samples with highest loss (hard examples)")
print("  4. Train on these hard examples")
print()
print("Benefits:")
print("  ✓ Focus computation on difficult samples")
print("  ✓ Can improve accuracy")
print("  ✓ Particularly useful for object detection")


## 8. Memory-Efficient Data Loading


In [ ]:
# Tips for memory efficiency
print("Memory-Efficient Data Loading Tips:")
print("="*70)
print()

print("1. USE APPROPRIATE NUM_WORKERS")
print("   • Start with num_workers=4")
print("   • Increase if CPU is bottleneck")
print("   • Too many workers can slow down due to overhead")
print()

print("2. PIN MEMORY")
print("   • Set pin_memory=True for CUDA")
print("   • Speeds up CPU-to-GPU transfer")
print("   loader = DataLoader(dataset, pin_memory=True)")
print()

print("3. USE NON_BLOCKING TRANSFERS")
print("   data = data.to(device, non_blocking=True)")
print("   • Overlaps data transfer with computation")
print()

print("4. OPTIMIZE BATCH SIZE")
print("   • Larger batches = better GPU utilization")
print("   • But limited by GPU memory")
print("   • Use gradient accumulation if needed")
print()

print("5. REDUCE DATALOADER OVERHEAD")
print("   • Use persistent_workers=True (PyTorch 1.7+)")
print("   • Prefetch_factor to control prefetching")
print()

print("6. COMPRESS OR DOWNSAMPLE IMAGES")
print("   • Store smaller images on disk")
print("   • Resize during loading only if needed")
print()

# Example: Optimized DataLoader
optimized_loader = DataLoader(
    dataset=None,  # your dataset
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,  # Reuse workers
    prefetch_factor=2          # Prefetch 2 batches
)

print("\nOptimized DataLoader configuration shown above!")


## 9. Advanced: DALI (NVIDIA Data Loading Library)


In [ ]:
print("NVIDIA DALI (Data Loading Library):")
print("="*70)
print()
print("What is DALI?")
print("  • GPU-accelerated data loading and augmentation")
print("  • Can provide 2-5x speedup over PyTorch DataLoader")
print("  • Particularly beneficial for image data")
print()
print("When to use DALI:")
print("  ✓ Training on NVIDIA GPUs")
print("  ✓ Image/video data")
print("  ✓ Data loading is bottleneck")
print("  ✗ Complex custom transforms")
print()
print("Installation:")
print("  pip install --extra-index-url https://developer.download.nvidia.com/compute/redist --upgrade nvidia-dali-cuda110")
print()
print("Example usage (requires DALI installation):")
print("""
from nvidia.dali import pipeline_def
import nvidia.dali.fn as fn
import nvidia.dali.types as types

@pipeline_def
def simple_pipeline():
    jpegs, labels = fn.readers.file(file_root='./data')
    images = fn.decoders.image(jpegs, device='mixed')
    images = fn.resize(images, size=[224, 224])
    images = fn.crop_mirror_normalize(images, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    return images, labels
""")
print()
print("For most use cases, PyTorch DataLoader is sufficient!")


## 📝 Summary

✅ Implemented advanced data augmentation  
✅ Applied Mixup and CutMix  
✅ Handled imbalanced datasets  
✅ Optimized data loading with prefetching  
✅ Cached expensive preprocessing  
✅ Handled multi-modal data  
✅ Implemented OHEM  
✅ Learned memory-efficient loading  

### Data Processing Checklist

**Before Training:**
- [ ] Analyze class distribution
- [ ] Choose appropriate augmentations
- [ ] Set up efficient data pipeline
- [ ] Configure num_workers and pin_memory
- [ ] Handle class imbalance if present

**During Training:**
- [ ] Monitor data loading time
- [ ] Check GPU utilization
- [ ] Adjust batch size and num_workers
- [ ] Use prefetching if needed

**Advanced Techniques:**
- [ ] Try Mixup/CutMix for regularization
- [ ] Implement caching for slow datasets
- [ ] Use OHEM for hard examples
- [ ] Consider DALI for image-heavy workflows

### Performance Tips

| Technique | Speedup | When to Use |
|-----------|---------|-------------|
| pin_memory=True | 5-10% | Always with CUDA |
| Prefetching | 10-30% | Data loading bottleneck |
| num_workers=4-8 | 20-50% | I/O intensive |
| Caching | 2-5x | Small datasets, slow I/O |
| DALI | 2-5x | Image data, NVIDIA GPUs |

### Next Steps

- **Complete!** You've finished the intermediate section! 🎉
- **Continue**: Move to `advanced/01_transformers.ipynb`
- **Practice**: Optimize your data pipeline
- **Challenge**: Implement all techniques on a real project


## 🎯 Practice Exercises


In [ ]:
# Exercise 1: Implement custom augmentation pipeline
# Your code here:


# Exercise 2: Create a balanced sampler for multi-class imbalanced dataset
# Your code here:


# Exercise 3: Implement data prefetching and measure speedup
# Your code here:


# Exercise 4: Build a caching dataset with LRU cache
# Your code here:


# Exercise 5: Create a multi-modal dataset (image + text + numerical features)
# Your code here:
